# MuSeg-AI Thigh Segmentation — Sheffield Dataset (Lambda)

Runs [fabianbalsiger/museg-ai](https://github.com/fabianbalsiger/museg-ai) (`thigh-model3`)
on the 69 Sheffield augmented DICOM volumes.

Sheffield provides a single greyscale image (no Dixon Fat channel).
MuSeg expects two channels (in-phase, out-of-phase); the same greyscale image
is passed as both channels (water-only mode: `[img, img]`).

Data: `~/sheffeld/20440164/Aug_N.dcm`
Output: `~/museg_sheffield_segs/Aug_N_dseg.nii.gz`

## 1 — Upload to Lambda
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/sheffeld \
  ubuntu@<YOUR-LAMBDA-IP>:~/
```

## 2 — Download results when done
```bash
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<YOUR-LAMBDA-IP>:~/museg_sheffield_segs/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/museg/sheffield_segs/
```
**Terminate the instance when done.**

In [ ]:
import subprocess, sys

def sh(cmd):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    out = (r.stdout + r.stderr).strip()
    if r.returncode != 0:
        print(f'[WARN] {cmd[:80]}: {out[:300]}')
    else:
        print(f'OK: {cmd[:60]}')
    return r.returncode == 0

r = subprocess.run('which docker', shell=True, capture_output=True)
if r.returncode != 0:
    sh('sudo apt-get update -qq')
    sh('sudo apt-get install -y docker.io')
else:
    print('Docker present:', r.stdout.strip())

sh('sudo systemctl start docker')
sh('sudo chmod 666 /var/run/docker.sock')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'git+https://github.com/fabianbalsiger/museg-ai.git',
    'pydicom', 'SimpleITK'])

import importlib; importlib.invalidate_caches()
import musegai
print('museg-ai version:', musegai.__version__)

In [ ]:
import docker
try:
    client = docker.from_env()
    client.ping()
    print('Docker is running.')
except Exception as e:
    raise RuntimeError(f'Docker not reachable — re-run setup cell.\n{e}')

In [ ]:
import glob, os, re
import numpy as np
import SimpleITK as sitk
import pydicom
from musegai import api
from musegai.api import Volume

IMG_DIR    = os.path.expanduser('~/sheffeld/20440164')
NII_DIR    = os.path.expanduser('~/sheffeld_nii')         # temp NIfTI cache
OUTPUT_DIR = os.path.expanduser('~/museg_sheffield_segs')

os.makedirs(NII_DIR,    exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

LABEL_MAP = {
    1:  'Vastus_Lateralis',   2:  'Vastus_Intermedius',
    3:  'Vastus_Medialis',    4:  'Rectus_Femoris',
    5:  'Sartorius',          6:  'Gracilis',
    7:  'Semimembranosus',    8:  'Semitendinosus',
    9:  'Biceps_Femoris',     10: 'Biceps_Femoris_Short',
    11: 'Adductor_Magnus',    12: 'Adductor_Longus',
    13: 'Adductor_Brevis',
}

dcm_files = sorted(
    [f for f in glob.glob(os.path.join(IMG_DIR, 'Aug_*.dcm'))
     if '_segmentations' not in f],
    key=lambda p: int(re.search(r'Aug_(\d+)\.dcm', p).group(1)),
)
print(f'Found {len(dcm_files)} DICOM volumes')


def dicom_to_nifti(dcm_path, nii_path):
    if os.path.exists(nii_path):
        return
    ds  = pydicom.dcmread(dcm_path)
    arr = ds.pixel_array.astype(np.float32)
    ps  = getattr(ds, 'PixelSpacing', [1.0, 1.0])
    st  = float(getattr(ds, 'SliceThickness', 1.0))
    img = sitk.GetImageFromArray(arr)
    img.SetSpacing([float(ps[1]), float(ps[0]), st])
    sitk.WriteImage(sitk.Cast(img, sitk.sitkFloat32), nii_path)

In [ ]:
for dcm_path in dcm_files:
    idx      = re.search(r'Aug_(\d+)\.dcm', dcm_path).group(1)
    nii_path = os.path.join(NII_DIR, f'Aug_{idx}.nii.gz')
    out_path = os.path.join(OUTPUT_DIR, f'Aug_{idx}_dseg.nii.gz')

    if os.path.exists(out_path):
        print(f'Skipping (done): Aug_{idx}')
        continue

    print(f'\nProcessing: Aug_{idx}')
    dicom_to_nifti(dcm_path, nii_path)

    img_vol = Volume.load(nii_path)
    print(f'  Shape: {img_vol.shape}  Spacing: {img_vol.spacing}')

    # Water-only mode: pass the same volume as both input channels
    results, labels = api.segment_volumes(
        {f'Aug_{idx}': [img_vol, img_vol]},
        model='thigh-model3',
        side='left+right',
    )

    segmentation = results[f'Aug_{idx}']
    segmentation.save(out_path)

    seg_arr = segmentation.array
    print(f'  Labels: {sorted(np.unique(seg_arr).tolist())}')
    for idx_lbl, name in LABEL_MAP.items():
        n = int((seg_arr == idx_lbl).sum())
        if n > 0:
            print(f'    {idx_lbl:<4} {name:<25} {n:>10,}')
    print(f'  Saved → {out_path}')

print('\nAll done.')

In [ ]:
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*_dseg.nii.gz')))
print(f'Output files: {len(results)} / {len(dcm_files)}')
if results:
    sample = Volume.load(results[0])
    print(f'Sample : {results[0]}')
    print(f'Shape  : {sample.shape}')
    print(f'Labels : {sorted(np.unique(sample.array).tolist())}')